# 第三部分 3.3：循环神经网络（RNN）

| 章节 | 内容 |
|------|------|
| **3.5 循环神经网络** | 序列建模、梯度消失、LSTM / GRU |

---

## 3.5 循环神经网络（RNN）

### 序列数据的特殊性

CNN 和 MLP 的输入假设：**样本之间相互独立**，每张图像与其他图像无关。

但很多真实数据是**序列**——当前的值依赖于之前的历史：
- 文本：「我喜欢__」，下一个词依赖前面的句子结构
- 时间序列：今天的股价依赖过去几天的走势
- 语音：当前的音频帧依赖前面的发音
- 视频：当前帧依赖前序帧

CNN / MLP 处理序列的困境：
- **输入长度固定**：这些模型的输入维度在设计时就固定了，无法处理任意长度的序列
- **无顺序感知**：MLP 把序列展平，丢失了「谁在谁前面」的信息
- **不共享时间权重**：第 1 步和第 5 步的模式识别应该用同一套参数，MLP 对每个位置用不同权重

---

### RNN 的基本结构

RNN 的核心想法：维护一个**隐藏状态（Hidden State）** $h_t$，作为「记忆」，在每个时间步更新：

$$h_t = \tanh(W_h h_{t-1} + W_x x_t + b)$$

$$y_t = W_y h_t + b_y$$

- $x_t$：当前时间步的输入
- $h_{t-1}$：上一时间步的隐藏状态（记忆）
- $h_t$：当前时间步的新隐藏状态
- $y_t$：当前时间步的输出（可选，看任务需要）

**关键**：$W_h, W_x, b$ 在所有时间步共享——无论序列多长，参数数量不变。

```
x₁ → [RNN Cell] → h₁ → [RNN Cell] → h₂ → [RNN Cell] → h₃ → 输出
           ↑                  ↑                  ↑
         h₀=0              h₁               h₂

同一套参数 W_h, W_x 在每个时间步复用
```

---

### RNN 的输入与输出：具体是什么？

**$x_t$ 是一个向量**，不是字符串或整数。以文本为例，模型不能直接处理汉字或英文单词，需要先通过 **Embedding 层**把词转成浮点向量：

```
原始文本：\"我爱机器学习\"
  ↓ 分词
词列表：[\"我\", \"爱\", \"机器\", \"学习\"]
  ↓ 词表查询（每个词 → 整数 ID）
词 ID：[3, 7, 25, 48]
  ↓ Embedding 层（一张可学习的查找表，形状：vocab_size × embed_dim）
词向量：[v₃, v₇, v₂₅, v₄₈]    每个 vᵢ ∈ ℝ^embed_dim（如 128 维浮点向量）

RNN 每步接收一个词向量 xₜ，依次处理 4 步
```

- **Embedding 层**本质是一张查找表：第 $i$ 行就是词 $i$ 的向量，整个表在训练中被学习
- 对于时间序列（如股价、传感器读数），$x_t$ 直接是数值向量，无需 Embedding

**$h_t$ 在每步都被计算**，但不同任务使用不同位置的输出：

```
时间步：  t=1        t=2        t=3        t=4（最后一步）
输入：    x₁  →     x₂  →     x₃  →     x₄
隐状态：  h₁         h₂         h₃         h₄

Many-to-One：                              h₄ → 分类头 → 标签
             只用最后一步，因为 h₄ 已积累了整个序列的信息

Many-to-Many：h₁→标签₁, h₂→标签₂, h₃→标签₃, h₄→标签₄
              每步都产生输出，适合输入输出等长的序列标注任务
```

---

### RNN 的梯度消失问题

RNN 的训练通过**时序反向传播（BPTT, Backpropagation Through Time）** 完成——将 RNN 在时间上「展开」成一个深层网络，然后用普通反向传播计算梯度。

**问题**：展开后的网络可能有几百层（长序列有多少步就展开多少层）。梯度从最后一步反传到最初一步，需要经过所有时间步的矩阵乘法。如果 $\|W_h\| < 1$，梯度指数级消失；如果 $\|W_h\| > 1$，梯度指数级爆炸。

实际影响：**RNN 很难学到长距离依赖**。「它早年学过钢琴，所以他后来的__」——「他」依赖序列开头的「它」，但这种跨越很多步的依赖，普通 RNN 几乎无法学到。

---

### LSTM：门控记忆

LSTM（Long Short-Term Memory，1997）通过引入**门控机制（Gating）** 和一条独立的**细胞状态（Cell State）** 来解决梯度消失问题。

细胞状态 $C_t$ 是 LSTM 的核心——它像一条「传送带」，信息可以直接从头传到尾，几乎不经过非线性变换，梯度可以几乎无损地反向传播。

**三个门控制细胞状态的读写：**

$$f_t = \sigma(W_f [h_{t-1}, x_t] + b_f) \qquad \text{（遗忘门：该忘掉多少旧记忆）}$$

$$i_t = \sigma(W_i [h_{t-1}, x_t] + b_i), \quad \tilde{C}_t = \tanh(W_C [h_{t-1}, x_t] + b_C) \qquad \text{（输入门：该写入多少新信息）}$$

$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t \qquad \text{（更新细胞状态：遗忘旧的 + 写入新的）}$$

$$o_t = \sigma(W_o [h_{t-1}, x_t] + b_o), \quad h_t = o_t \odot \tanh(C_t) \qquad \text{（输出门：该输出多少）}$$

其中 $\odot$ 是逐元素乘法（Hadamard 积），$\sigma$ 是 Sigmoid 函数（输出 0~1，控制门开/关程度）。

**直觉类比**：
- **遗忘门**：「我读到第 10 句，发现主语换了，之前记住的性别信息可以忘掉了」
- **输入门**：「这一步出现了重要词汇，我要把它写入长期记忆」
- **输出门**：「当前任务需要读取细胞状态里的哪部分来形成隐藏状态」

---

### GRU：简化版 LSTM

GRU（Gated Recurrent Unit，2014）把 LSTM 的三个门简化为两个，并合并细胞状态和隐藏状态：

$$z_t = \sigma(W_z [h_{t-1}, x_t]) \qquad \text{（更新门，控制新旧信息比例）}$$

$$r_t = \sigma(W_r [h_{t-1}, x_t]) \qquad \text{（重置门，控制旧记忆的读取程度）}$$

$$\tilde{h}_t = \tanh(W [r_t \odot h_{t-1}, x_t]) \qquad \text{（候选新状态）}$$

$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t \qquad \text{（线性插值：保留旧的 + 写入新的）}$$

| | RNN | LSTM | GRU |
|-|-----|------|-----|
| 状态 | $h_t$ | $h_t$ + $C_t$ | $h_t$ |
| 门数量 | 0 | 3 个门 | 2 个门 |
| 参数量 | 最少 | 最多（约 4× RNN）| 中等（约 3× RNN）|
| 长程依赖 | 差 | 好 | 好（略逊于 LSTM）|
| 训练速度 | 快 | 慢 | 中等 |

> **实践建议**：
> - 优先尝试 **GRU**（参数更少，训练更快，性能接近 LSTM）
> - 需要极高精度或超长序列时用 **LSTM**
> - 在 NLP 中，大多数任务已被 **Transformer** 取代，LSTM/GRU 主要用于时间序列、嵌入式设备等对推理速度有严格要求的场景

---

### RNN 的任务类型与真实应用

RNN（以及它的改进版 LSTM / GRU）可以适配四种不同的输入输出结构：

**多对一（Many-to-One）**：读入整个序列，输出单个结果。
```
x₁ → x₂ → x₃ → x₄
h₁    h₂    h₃    h₄
                   ↓
                 输出
```
- 情感分类：读完一段影评，输出正面 / 负面
- 垃圾邮件检测：读完一封邮件，输出是 / 否
- 股价涨跌预测：读入过去 30 天收盘价，预测明天涨 / 跌

通常用单个 LSTM，取最后一步的 $h_T$ 接分类头。

---

**多对多，等长（Many-to-Many）**：序列进、序列出，每步对应一个输出。
```
x₁ → x₂ → x₃ → x₄
h₁    h₂    h₃    h₄
↓     ↓     ↓     ↓
标签₁ 标签₂ 标签₃ 标签₄
```
- 命名实体识别（NER）：「苹果 公司 发布 新品」→「机构 机构 O O」，每个词输出一个标签
- 词性标注：每个词对应一个词性（名词 / 动词 / 形容词…）

单个 LSTM，每个时间步的 $h_t$ 都接一个输出头。

---

**一对多（One-to-Many）**：输入单个向量，逐步生成一个序列。
```
c（初始向量）
    ↓
    h₀ → y₁ → h₁ → y₂ → h₂ → y₃ → ...
```
- 图像描述生成（Image Captioning）：CNN 提取图像特征向量 $c$，LSTM Decoder 逐词生成描述文字

单个 LSTM 当 Decoder，以 $c$ 作为初始隐状态启动。

---

**多对多，变长（Seq2Seq）**：输入序列和输出序列长度不同、语序可能不同。
```
Encoder LSTM          Decoder LSTM
x₁→x₂→x₃→x₄ → c → s₀→y₁→s₁→y₂→s₂→y₃→...
```
- 机器翻译：「I love ML」→「我爱机器学习」，英文 3 词对应中文 5 字
- 文本摘要：长文章 → 短摘要
- 对话系统：用户输入 → 模型回复

实际上**几乎所有 Seq2Seq 都用 LSTM + LSTM**，而不是普通 RNN。原因是 Encoder 需要把整句话压缩进 $c$，这本身就极度依赖长程记忆；普通 RNN 因梯度消失，$c$ 里基本只剩最后几个词的信息，前半句就丢掉了。Sutskever 2014 年提出 Seq2Seq 的原始论文用的就是 LSTM Encoder + LSTM Decoder。